# Lab 04 - Frame It Three Ways, Then Guard the Contract

**Week 3 · Prompt Engineering and Task-to-Prompt Mapping**
**Difficulty:** Intermediate · **Time:** ~150 minutes · **Runs fully offline (no network, no API key).**

You are helping Finance Ops at a fictional retailer, **Fernwood Hardware**, triage a week of
travel-expense messages and produce a manager digest. One vague stakeholder request:

> "Can you look at last week's travel expense emails, sort them, pull the key fields, and give me a quick summary for managers?"

That single sentence hides **three different tasks**. In this lab you will:

1. **Frame it three ways** - classification, extraction (schema-first), and summarization - each as a precise, delimited prompt with an explicit output contract.
2. **Guard the contract** - encode the extraction contract as a JSON Schema, validate model output, generate a repair instruction from the errors, and repair to a clean result.
3. **Report verified numbers** - build the manager digest from the clean data.

**Why offline.** Live model output is non-reproducible, so this lab replaces the network call with a
deterministic `simulate_model(...)` stub that returns realistic, contract-violating output. The engineering
you practice (framing, schema guardrails, validate-repair loops) is exactly what you would wrap around a
real endpoint. An appendix shows how to swap the stub for a live Anthropic or OpenAI call.

**How to work this notebook.** Cells with `# TODO` are yours to complete. Each is followed by a `check(...)`
cell. The notebook opens mostly **red** and is done when every check reads **PASS**. Hints are deliberately
sparse; if you get stuck, open `HINTS.md`.

## Part A - Corpus and Model Stub (provided)

Run the next two cells as-is. They define the policy, the six messages, the reference schema fields,
and the deterministic model stub. You do not edit these.

In [ ]:
%pip install -r requirements.txt

In [1]:
from importlib.metadata import version
print("jsonschema", version("jsonschema"))

POLICY = """1. Eligible categories: airfare, lodging, ground_transport, meals, other.
2. Receipts required for any single expense >= 25.00 USD.
3. Tips are allowed up to 20 percent and must be itemized.
4. For foreign currency, record original currency and USD converted total.
5. Missing receipt: mark status 'needs_proof' and request re-submission."""

SNIPPETS = {
 "E-101": "Took Uber from SFO to hotel, USD 42.85, tip 15%, receipt attached.",
 "E-102": "Hotel in Austin 3 nights, total $612.44. I think the receipt is somewhere.",
 "E-103": "Flight AA1421 SFO to AUS $389.60. Fare + taxes included. PDF receipt included.",
 "E-104": "Team dinner: $128.90 for 4 people at 'Oak and Rye'. Forgot to keep receipt.",
 "E-105": "Metro card in Austin about $25. Cash. No receipt, sorry.",
 "E-106": "Taxi from AUS to client site 38.00 USD, no tip. Photo of receipt included.",
}
VALID_CATEGORIES = ["airfare", "lodging", "ground_transport", "meals", "other"]
ALLOWED_KEYS = {"input_id","category","amount_usd","tip_percent","currency",
                "has_receipt","status","original_currency","notes"}
CATEGORY_ALIASES = {"hotel":"lodging","flight":"airfare","air":"airfare","cab":"ground_transport",
                    "taxi":"ground_transport","uber":"ground_transport","rideshare":"ground_transport",
                    "food":"meals","dinner":"meals"}
print("corpus loaded:", len(SNIPPETS), "messages")

jsonschema 4.26.0
corpus loaded: 6 messages


In [2]:
import copy

# Deterministic stand-in for "paste this prompt into your model and read back JSON".
# The first pass returns realistic, contract-violating output. A stronger (few-shot)
# prompt slips fewer flaws. This is the ONLY thing standing in for a live model.
_MODEL_TRUTH = [
 {"input_id":"E-101","category":"ground_transport","amount_usd":42.85,"tip_percent":15.0,
  "currency":"USD","has_receipt":True,"status":"ok","original_currency":None,
  "notes":"Uber to hotel; receipt attached; tip within policy."},
 {"input_id":"E-102","category":"lodging","amount_usd":612.44,"tip_percent":None,
  "currency":"USD","has_receipt":False,"status":"needs_proof","original_currency":None,
  "notes":"Hotel 3 nights; receipt uncertain; needs proof."},
 {"input_id":"E-103","category":"airfare","amount_usd":389.60,"tip_percent":None,
  "currency":"USD","has_receipt":True,"status":"ok","original_currency":None,
  "notes":"Flight; PDF receipt included."},
 {"input_id":"E-104","category":"meals","amount_usd":128.90,"tip_percent":None,
  "currency":"USD","has_receipt":False,"status":"needs_proof","original_currency":None,
  "notes":"Team dinner; no receipt kept; needs proof."},
 {"input_id":"E-105","category":"ground_transport","amount_usd":25.00,"tip_percent":None,
  "currency":"USD","has_receipt":False,"status":"needs_proof","original_currency":None,
  "notes":"Metro card; cash; no receipt; at 25 threshold."},
 {"input_id":"E-106","category":"ground_transport","amount_usd":38.00,"tip_percent":None,
  "currency":"USD","has_receipt":True,"status":"ok","original_currency":None,
  "notes":"Taxi to client site; photo receipt; no tip."},
]

def simulate_model(task, few_shot=False):
    """Deterministic offline model. task='extract' returns a list of records.
    Zero-shot output carries six realistic flaws; few_shot=True carries one."""
    if task != "extract":
        raise ValueError(f"this stub only supports task='extract', got {task!r}")
    raw = copy.deepcopy(_MODEL_TRUTH)
    if not few_shot:
        raw[0]["tip_percent"] = "15%"       # wrong type: string, not number
        raw[0]["confidence"] = 0.91         # extra key not in contract
        raw[1]["category"] = "hotel"        # enum miss: should be lodging
        raw[2]["input_id"] = "E103"         # pattern miss: missing hyphen
        del raw[3]["has_receipt"]           # missing required key
        raw[4]["status"] = "ok"             # policy miss: should be needs_proof
        raw[5]["amount_usd"] = "38.00"      # wrong type: string, not number
    else:
        raw[1]["category"] = "hotel"        # a stronger prompt still slips one
    return raw

print("model stub ready. zero-shot sample record 0:")
print(simulate_model("extract")[0])

model stub ready. zero-shot sample record 0:
{'input_id': 'E-101', 'category': 'ground_transport', 'amount_usd': 42.85, 'tip_percent': '15%', 'currency': 'USD', 'has_receipt': True, 'status': 'ok', 'original_currency': None, 'notes': 'Uber to hotel; receipt attached; tip within policy.', 'confidence': 0.91}


### The `check(...)` helper

`check` is a soft assertion: it prints PASS or FAIL and never stops the notebook. Aim to turn every
check green. Run this cell before working the TODOs.

In [3]:
def check(label, condition, detail=""):
    mark = "\u2705" if condition else "\u274c"
    status = "PASS" if condition else "FAIL"
    line = f"{mark} {status}  {label}"
    if detail and not condition:
        line += f"\n        -> {detail}"
    print(line)
    return bool(condition)

def _has_all(text, tokens):
    t = (text or "").lower()
    missing = [tok for tok in tokens if tok.lower() not in t]
    return (not missing), missing

check("check() helper is live", True)

✅ PASS  check() helper is live


True

## Part B - Frame It Three Ways

The same messy request maps to three distinct prompt designs. You will author each as a template string.
These are graded **structurally** (does the prompt carry the elements a reliable prompt needs), not by
calling a model. A good frame has: a role, delimited context, explicit constraints, an output contract,
and a self-check cue.

Use `{policy}` as a placeholder where the policy text belongs (do not paste the whole policy inline).

### B1 - Classification frame (triage: one label + a compliance status)

In [4]:
# TODO 1: Author the classification prompt.
# Contract: a single string. It must establish a role, include the policy as delimited
# context via the {policy} placeholder, list the five labels and the two statuses, state
# constraints (one label; needs_proof when amount >= 25 USD and receipt missing; cite policy
# lines; notes 8 to 24 words), give a JSON output contract, and end with a self-check cue.
CLASSIFY_PROMPT = ""  # replace with your template string


In [5]:
ok, miss = _has_all(CLASSIFY_PROMPT, ["policy","json","status","needs_proof","self"])
check("CLASSIFY_PROMPT has role/context/contract/self-check", ok, f"missing tokens: {miss}")
check("CLASSIFY_PROMPT uses a delimiter (### or <<< or ```)",
      any(d in CLASSIFY_PROMPT for d in ["<<<", "###", "```"]))
check("CLASSIFY_PROMPT lists all five labels",
      all(l in CLASSIFY_PROMPT.lower() for l in VALID_CATEGORIES))
check("CLASSIFY_PROMPT keeps policy out-of-line via {policy}", "{policy}" in CLASSIFY_PROMPT)

❌ FAIL  CLASSIFY_PROMPT has role/context/contract/self-check
        -> missing tokens: ['policy', 'json', 'status', 'needs_proof', 'self']
❌ FAIL  CLASSIFY_PROMPT uses a delimiter (### or <<< or ```)
❌ FAIL  CLASSIFY_PROMPT lists all five labels
❌ FAIL  CLASSIFY_PROMPT keeps policy out-of-line via {policy}


False

### B2 - Extraction frame (schema-first, for Finance ingestion)

In [6]:
# TODO 2: Author the extraction prompt.
# Contract: a single string. It must establish a role, include the policy via {policy},
# describe the schema fields with types and which are required, forbid invented values,
# require numeric fields to be plain numbers (no symbols or percent signs), require a
# VALID JSON array with no extra keys, and end with a self-verify cue.
EXTRACT_PROMPT = ""  # replace with your template string


In [7]:
ok, miss = _has_all(EXTRACT_PROMPT, ["schema","json","invent","required","self"])
check("EXTRACT_PROMPT has schema/contract/no-invention/self-verify", ok, f"missing tokens: {miss}")
check("EXTRACT_PROMPT uses a delimiter",
      any(d in EXTRACT_PROMPT for d in ["<<<", "###", "```"]))
check("EXTRACT_PROMPT keeps policy out-of-line via {policy}", "{policy}" in EXTRACT_PROMPT)

❌ FAIL  EXTRACT_PROMPT has schema/contract/no-invention/self-verify
        -> missing tokens: ['schema', 'json', 'invent', 'required', 'self']
❌ FAIL  EXTRACT_PROMPT uses a delimiter
❌ FAIL  EXTRACT_PROMPT keeps policy out-of-line via {policy}


False

### B3 - Summarization frame (manager digest)

In [8]:
# TODO 3: Author the summarization prompt.
# Contract: a single string aimed at Finance managers. It must state audience and scope,
# constrain length (at most 6 bullets, at most 14 words each), forbid PII and blaming
# language, give the Markdown digest contract (total spend, category split, items needing
# proof, policy risks, quick actions), and end with a self-check on the total and flags.
SUMMARIZE_PROMPT = ""  # replace with your template string


In [9]:
ok, miss = _has_all(SUMMARIZE_PROMPT, ["manager","total spend","proof","bullet"])
check("SUMMARIZE_PROMPT targets managers with a digest contract", ok, f"missing tokens: {miss}")
check("SUMMARIZE_PROMPT constrains length (mentions 6 and 14)",
      ("6" in SUMMARIZE_PROMPT and "14" in SUMMARIZE_PROMPT))

❌ FAIL  SUMMARIZE_PROMPT targets managers with a digest contract
        -> missing tokens: ['manager', 'total spend', 'proof', 'bullet']
❌ FAIL  SUMMARIZE_PROMPT constrains length (mentions 6 and 14)


False

## Part C - Guard the Contract (schema, validate, repair)

Prompts alone do not guarantee well-formed output; a capable model still returns the occasional bad record.
So we enforce the contract in code. You will build the schema, a validator, a repair-prompt generator, and a
deterministic repair step, then run the validate-repair-revalidate loop against the stub's flawed output.

### C1 - JSON Schema (Draft 2020-12)

Encode the extraction contract as a Draft 2020-12 schema for an **array of objects**.

In [10]:
# TODO 4: Build EXPENSE_SCHEMA as a Draft 2020-12 schema.
# Contract: an array whose items are objects. Required keys: input_id, category,
# amount_usd, has_receipt, status. Constrain input_id to the pattern ^E-###
# (three digits), category and status to their enums, amount_usd to a non-negative
# number, tip_percent to a number in 0..20 or null, has_receipt to boolean, notes to
# a string (or null) capped at 160 chars. Reject any properties not in the contract.
EXPENSE_SCHEMA = {}  # replace with your schema dict


In [11]:
from jsonschema import Draft202012Validator
try:
    Draft202012Validator.check_schema(EXPENSE_SCHEMA)
    check("EXPENSE_SCHEMA is itself a valid Draft 2020-12 schema", True)
    good = [{"input_id":"E-999","category":"meals","amount_usd":10.0,
             "has_receipt":True,"status":"ok"}]
    bad_extra = [dict(good[0], nope=1)]
    v = Draft202012Validator(EXPENSE_SCHEMA)
    check("schema accepts a well-formed record", v.is_valid(good))
    check("schema rejects an unexpected extra key", not v.is_valid(bad_extra))
    check("schema rejects a bad input_id pattern",
          not v.is_valid([dict(good[0], input_id="E999")]))
    check("schema rejects an out-of-range tip_percent",
          not v.is_valid([dict(good[0], tip_percent=150)]))
except Exception as e:
    check("EXPENSE_SCHEMA usable", False, repr(e))

✅ PASS  EXPENSE_SCHEMA is itself a valid Draft 2020-12 schema
✅ PASS  schema accepts a well-formed record
❌ FAIL  schema rejects an unexpected extra key
❌ FAIL  schema rejects a bad input_id pattern
❌ FAIL  schema rejects an out-of-range tip_percent


### C2 - Validator

Wrap the schema in a function that returns a **sorted list of readable error strings** (empty when valid).

In [12]:
# TODO 5: Implement validate(data, schema=EXPENSE_SCHEMA).
# Contract: return a list of strings, one per schema violation, sorted for stable output,
# each formatted as "<json_path>: <message>". Return [] when data fully conforms.
def validate(data, schema=None):
    raise NotImplementedError("implement validate()")


In [13]:
try:
    raw = simulate_model("extract")
    errs = validate(raw)
    check("validate() returns a list", isinstance(errs, list))
    check("validate() flags the six seeded structural errors on raw output",
          len(errs) == 6, f"got {len(errs)}: {errs}")
    check("validate() returns [] for a clean record",
          validate([{"input_id":"E-100","category":"other","amount_usd":1.0,
                     "has_receipt":True,"status":"ok"}]) == [])
    print("\nseeded errors:")
    for e in errs: print("  ", e)
except NotImplementedError as e:
    check("validate() implemented", False, str(e))
except Exception as e:
    check("validate() runs without crashing", False, repr(e))

❌ FAIL  validate() implemented
        -> implement validate()


> **Notice what is *not* in that list.** The stub set `E-105` to `status: "ok"`, which is a legal enum
> value, so the schema passes it. Schema validity is not the same as policy correctness. The repair step
> below re-derives status from policy and catches it.

### C3 - Repair-prompt generator

If you were sending output back to a real model to fix, you would hand it the errors plus the contract rules.
Build that instruction text from a list of errors.

In [14]:
# TODO 6: Implement make_repair_prompt(errors, schema_name="schemas/expense.schema.json").
# Contract: return a single string that (a) tells the model to repair to the named contract,
# (b) states the repair rules (keep valid records, fix types/enums, coerce tip to 0..20 or
# null, input_id ^E-###, recompute status from policy, drop extra keys, JSON array only),
# and (c) embeds the given validator errors. If errors is empty, embed "(none)".
def make_repair_prompt(errors, schema_name="schemas/expense.schema.json"):
    raise NotImplementedError("implement make_repair_prompt()")


In [15]:
try:
    p = make_repair_prompt(validate(simulate_model("extract")))
    check("repair prompt demands a JSON array only", "VALID JSON array ONLY" in p.upper() or "JSON ARRAY ONLY" in p.upper())
    check("repair prompt states the policy status rule", "needs_proof" in p)
    check("repair prompt embeds the validator errors", "additional properties" in p.lower())
    check("empty errors embed '(none)'", "(none)" in make_repair_prompt([]))
except NotImplementedError as e:
    check("make_repair_prompt() implemented", False, str(e))
except Exception as e:
    check("make_repair_prompt() runs", False, repr(e))

❌ FAIL  make_repair_prompt() implemented
        -> implement validate()


### C4 - Deterministic repair

In this offline lab, apply the repair in code (the same transforms the repair prompt asks the model to make).
This also teaches that many guardrails need no second model call at all.

In [16]:
# TODO 7: Implement repair_records(raw) -> list[dict].
# Contract: return new records that satisfy EXPENSE_SCHEMA and the policy. For each record:
#   - keep only keys in ALLOWED_KEYS
#   - normalize input_id to the form E-### (three digits)
#   - coerce amount_usd and tip_percent to numbers (None stays None); strip stray symbols
#   - map category through CATEGORY_ALIASES, lower-cased
#   - if has_receipt is missing, default to False; otherwise coerce to bool
#   - recompute status from policy: needs_proof if amount_usd >= 25 and not has_receipt, else ok
# Do not mutate the input.
import re
def repair_records(raw):
    raise NotImplementedError("implement repair_records()")


In [17]:
try:
    raw = simulate_model("extract")
    clean = repair_records(raw)
    check("repair_records() does not mutate its input", "confidence" in raw[0])
    check("repaired output passes the schema", validate(clean) == [],
          f"still invalid: {validate(clean)}")
    check("E-102 category alias hotel -> lodging",
          next(r for r in clean if r["input_id"]=="E-102")["category"] == "lodging")
    check("E-103 input_id normalized to E-103",
          any(r["input_id"]=="E-103" for r in clean))
    check("E-105 status corrected to needs_proof by policy",
          next(r for r in clean if r["input_id"]=="E-105")["status"] == "needs_proof")
    check("E-106 amount coerced to numeric 38.0",
          next(r for r in clean if r["input_id"]=="E-106")["amount_usd"] == 38.0)
except NotImplementedError as e:
    check("repair_records() implemented", False, str(e))
except Exception as e:
    check("repair_records() runs", False, repr(e))

❌ FAIL  repair_records() implemented
        -> implement repair_records()


## Part D - Manager Digest and the Full Loop

Now consume the clean records to produce the manager numbers, then run the whole pipeline end to end.

In [18]:
# TODO 8: Implement build_manager_summary(clean) -> dict.
# Contract: return {"total_spend_usd": <float rounded to 2>, "category_split": [<categories
# in first-appearance order, de-duplicated>], "needs_proof": [<input_ids whose status is
# needs_proof, in order>]}.
def build_manager_summary(clean):
    raise NotImplementedError("implement build_manager_summary()")


In [19]:
try:
    summary = build_manager_summary(repair_records(simulate_model("extract")))
    check("total spend is the verified 1236.79", summary["total_spend_usd"] == 1236.79,
          f"got {summary['total_spend_usd']}")
    check("category split is correct and de-duplicated",
          summary["category_split"] == ["ground_transport","lodging","airfare","meals"],
          f"got {summary['category_split']}")
    check("needs_proof set is E-102, E-104, E-105",
          summary["needs_proof"] == ["E-102","E-104","E-105"], f"got {summary['needs_proof']}")
    print("\nsummary:", summary)
except NotImplementedError as e:
    check("build_manager_summary() implemented", False, str(e))
except Exception as e:
    check("build_manager_summary() runs", False, repr(e))

❌ FAIL  build_manager_summary() implemented
        -> implement repair_records()


### The full validate-repair-revalidate loop

This provided cell ties everything together the way a production guardrail would.

In [20]:
def run_pipeline(few_shot=False, verbose=True):
    raw = simulate_model("extract", few_shot=few_shot)
    before = validate(raw)
    repair_prompt = make_repair_prompt(before)          # what we'd send a live model
    clean = repair_records(raw)                          # what we do offline
    after = validate(clean)
    summary = build_manager_summary(clean)
    if verbose:
        print(f"errors before repair: {len(before)}")
        print(f"errors after repair:  {len(after)}")
        print("digest:", summary)
    return {"before": before, "after": after, "clean": clean, "summary": summary,
            "repair_prompt": repair_prompt}

def triage_view(clean):
    return [{"input_id":r["input_id"],"label":r["category"],"status":r["status"]} for r in clean]

try:
    result = run_pipeline()
    check("pipeline ends with zero validator errors", result["after"] == [])
    check("pipeline reduced the error count", len(result["before"]) > len(result["after"]))
    print("\ntriage view:")
    for row in triage_view(result["clean"]): print("  ", row)
except Exception as e:
    check("run_pipeline() runs end to end", False, repr(e))

❌ FAIL  run_pipeline() runs end to end
        -> NotImplementedError('implement validate()')


## Stretch Goals (optional)

Both have graded checks. Solutions ship in the instructor solution notebook.

### Stretch 1 - Does a stronger prompt need fewer repairs?

The stub accepts `few_shot=True` to emulate a better-engineered prompt. Quantify the difference: count how
many validator errors each variant produces before repair, and confirm both still repair to the same clean
digest.

In [21]:
# TODO S1: Implement compare_prompt_variants() -> dict with keys
# "zero_shot_errors" and "few_shot_errors" (ints), each the number of validator errors
# BEFORE repair for that variant. Assert to yourself that both still repair clean.
def compare_prompt_variants():
    raise NotImplementedError("implement compare_prompt_variants()")


In [22]:
try:
    cmp = compare_prompt_variants()
    check("zero-shot produces more pre-repair errors than few-shot",
          cmp["zero_shot_errors"] > cmp["few_shot_errors"], f"got {cmp}")
    print("comparison:", cmp)
except NotImplementedError as e:
    check("compare_prompt_variants() implemented", False, str(e))
except Exception as e:
    check("compare_prompt_variants() runs", False, repr(e))

❌ FAIL  compare_prompt_variants() implemented
        -> implement compare_prompt_variants()


### Stretch 2 - A second guardrail layer: policy checks the schema cannot express

A schema cannot say "status must match the policy computed from amount and receipt." Build that second
layer. It should return violations for schema-valid-but-policy-wrong records and pass clean data.

In [23]:
# TODO S2: Implement policy_violations(records) -> list[str].
# Contract: for each record, the expected status is needs_proof if amount_usd >= 25 and not
# has_receipt, else ok. If the record's status differs, add a readable violation string.
# Also flag any non-null tip_percent outside 0..20. Return [] when all records comply.
def policy_violations(records):
    raise NotImplementedError("implement policy_violations()")


In [24]:
try:
    schema_ok_policy_wrong = [{"input_id":"E-201","category":"meals","amount_usd":50.0,
        "has_receipt":False,"status":"ok"}]
    check("that record passes the schema", validate(schema_ok_policy_wrong) == [])
    pv = policy_violations(schema_ok_policy_wrong)
    check("policy layer flags the schema-valid-but-policy-wrong record", len(pv) == 1, f"got {pv}")
    check("clean pipeline data has zero policy violations",
          policy_violations(repair_records(simulate_model("extract"))) == [])
    print("policy violations on the crafted record:", pv)
except NotImplementedError as e:
    check("policy_violations() implemented", False, str(e))
except Exception as e:
    check("policy_violations() runs", False, repr(e))

❌ FAIL  policy_violations() implemented
        -> implement validate()


## Part E - Wrap-Up

### Knowledge checks (discuss)
1. Why split this one request into classification, extraction, and summarization rather than asking for everything at once?
2. What did the JSON Schema catch that a hand-written `if` ladder would likely miss, and what did it *not* catch?
3. When would you re-prompt a model to repair versus fix the data in code? What are the trade-offs?

### Self-assessment rubric (0-2 each, target >= 10/12)
- Three delimited frames, each with a role, constraints, and an output contract.
- Schema encodes types, enums, pattern, and rejects extra keys.
- Validator returns readable, sorted errors and is empty on clean data.
- Repair prompt fully restates the contract and embeds the errors.
- Deterministic repair yields schema-clean, policy-correct records.
- Manager digest reports the verified total and the correct flags.

### CURRENCY FLAG - determinism controls
> `temperature=0` and `top_p=1` **reduce** variation but do **not** guarantee identical output on hosted
> endpoints; a `seed` is best-effort and provider-specific (OpenAI exposes one; the Anthropic Messages API
> does not), and reasoning models lock sampling parameters. Treat schema validation plus a repair loop as
> your real determinism guarantee, not the sampling knobs. Verify the current parameter surface for your
> provider at build time.

### Appendix - swapping the stub for a live model (do not run in class)
The stub mirrors a real call. To go live, replace `simulate_model` with a client call, keep the model id in a
config variable (never hard-code an unverified id), read the key from the environment, and feed the raw output
straight into the same `validate` / `make_repair_prompt` / re-validate loop. Confirm the current model id and
parameter names against your provider's docs before running.

In [25]:
# Appendix (reference only, not executed): live call shape.
# import os
# MODEL = os.environ["EXPENSE_MODEL"]          # e.g. a current Anthropic or OpenAI model id (verify!)
# text  = call_your_provider(MODEL, EXTRACT_PROMPT.format(policy=POLICY), corpus=SNIPPETS)
# import json; raw = json.loads(text)          # models can still emit non-JSON; guard this
# errs = validate(raw)
# if errs:
#     repair = make_repair_prompt(errs)        # send back to the model, or repair_records(raw) offline
print("Appendix is reference-only. Lab complete when every check above reads PASS.")

Appendix is reference-only. Lab complete when every check above reads PASS.
